# **Scrapping Data**

Scraping data adalah teknik otomatisasi menggunakan bot atau skrip komputer untuk mengekstrak dan mengambil data spesifik dari berbagai sumber seperti halaman website, basis data, atau aplikasi lalu menyimpannya ke dalam format terstruktur seperti spreadsheet atau CSV. 

Cara Kerja Scraping DataProses pengambilan data ini umumnya melalui tiga tahapan utama:
- Request: Program mengirimkan permintaan (command) untuk mengakses halaman atau sumber data yang dituju.
- Parse: Sistem membaca struktur kode (seperti HTML) atau antarmuka untuk mencari dan mengidentifikasi data spesifik yang diinginkan.
- Display/Save: Data yang ditemukan diekstrak dan diubah menjadi format rapi seperti CSV, JSON, atau database agar mudah dianalisis.

## **Persiapan Library**
Memasang pustaka yang diperlukan (trafilatura untuk ekstraksi konten teks bersih, beautifulsoup4 untuk mengambil daftar tautan URL).

In [1]:
!pip install trafilatura beautifulsoup4 pandas requests

## **Langkah 1: Mengumpulkan 200 URL Artikel Detik.com**
Karena Trafilatura fokus mengekstrak isi halaman, kita perlu mengambil daftar URL terlebih dahulu dari halaman indeks berita Detik (misalnya kanal news). Kode ini akan melakukan paging otomatis hingga terkumpul minimal 100 URL unik

In [2]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

def get_detik_urls_by_category(base_url, target_count=100, category_name=""):
    urls = []
    page = 1
    
    print(f"Mengumpulkan URL untuk kategori: {category_name.upper()}...")
    while len(urls) < target_count:
        index_url = f"{base_url}?page={page}"
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        
        response = requests.get(index_url, headers=headers)
        if response.status_code != 200:
            print(f"Gagal mengakses halaman {page} untuk {category_name}")
            break
            
        soup = BeautifulSoup(response.text, 'html.parser')
        articles = soup.find_all('article')
        
        for article in articles:
            link = article.find('a')
            if link and 'href' in link.attrs:
                href = link['href']
                if "detik.com" in href and href not in [item['url'] for item in urls]:
                    urls.append({
                        'url': href,
                        'category': category_name
                    })
                    if len(urls) >= target_count:
                        break
                        
        print(f"Kategori {category_name} - Halaman {page}: Terkumpul {len(urls)} URL")
        page += 1
        time.sleep(1) # Jeda agar tidak diblokir server
        
    return urls[:target_count]

# 1. Kumpulkan 100 URL Sport
sport_urls = get_detik_urls_by_category(
    base_url="https://sport.detik.com/indeks", 
    target_count=100, 
    category_name="sport"
)

# 2. Kumpulkan 100 URL Finance
finance_urls = get_detik_urls_by_category(
    base_url="https://finance.detik.com/indeks", 
    target_count=100, 
    category_name="finance"
)

# Gabungkan menjadi satu list total 200 URL
all_target_urls = sport_urls + finance_urls
print(f"\nTotal keseluruhan URL terkumpul: {len(all_target_urls)} URL (100 Sport, 100 Finance).")

Mengumpulkan URL untuk kategori: SPORT...


Kategori sport - Halaman 1: Terkumpul 20 URL


Kategori sport - Halaman 2: Terkumpul 40 URL


Kategori sport - Halaman 3: Terkumpul 60 URL


Kategori sport - Halaman 4: Terkumpul 80 URL


...

Kategori finance - Halaman 2: Terkumpul 40 URL


Kategori finance - Halaman 3: Terkumpul 60 URL


Kategori finance - Halaman 4: Terkumpul 80 URL


Kategori finance - Halaman 5: Terkumpul 100 URL



Total keseluruhan URL terkumpul: 200 URL (100 Sport, 100 Finance).


## **Langkah 2: Proses Scrapping Menggunakan Trafilatura**
Setelah daftar URL terkumpul, iterasi setiap URL untuk mengekstrak judul, tanggal, penulis, dan teks isi berita secara bersih menggunakan fungsi trafilatura.

In [3]:
import trafilatura
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

data_hasil = []
total_data = len(all_target_urls)

print(f"Memulai scraping {total_data} data (Sport & Finance)...")
for idx, item in enumerate(all_target_urls):
    url = item['url']
    category = item['category']
    
    try:
        # 1. Ambil HTML mentah menggunakan requests untuk parsing judul & tanggal
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        resp = requests.get(url, headers=headers)
        
        title = None
        date = None
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, 'html.parser')
            # Ambil judul dari tag <h1> atau <title>
            title_tag = soup.find('h1') or soup.find('title')
            if title_tag:
                title = title_tag.text.strip()
                
            # Ambil tanggal dari tag div class detail__date
            date_tag = soup.find('div', class_='detail__date')
            if date_tag:
                date = date_tag.text.strip()

        # 2. Ambil teks bersih menggunakan Trafilatura
        downloaded = trafilatura.fetch_url(url)
        text = ""
        if downloaded:
            extracted = trafilatura.extract(downloaded, include_comments=False, include_tables=False)
            if extracted:
                text = extracted
                
        # Simpan ke list of dictionary
        data_hasil.append({
            'url': url,
            'category': category, # Menyimpan label sport atau finance
            'title': title,
            'date': date,
            'text': text
        })
        
        print(f"Berhasil scrape [{idx+1}/{total_data}] - {category.upper()}")
    except Exception as e:
        print(f"Gagal pada URL {url}: {e}")
        
    time.sleep(0.5)

# 3. Buat DataFrame
df = pd.DataFrame(data_hasil)

# 4. Tambahkan kolom panjang karakter dengan perlindungan fillna('') agar terhindar dari TypeError
df['title_length'] = df['title'].fillna('').astype(str).apply(len)
df['text_length'] = df['text'].fillna('').astype(str).apply(len)



Memulai scraping 200 data (Sport & Finance)...


Berhasil scrape [1/200] - SPORT


Berhasil scrape [2/200] - SPORT


Berhasil scrape [3/200] - SPORT


Berhasil scrape [4/200] - SPORT


...

Berhasil scrape [17/200] - SPORT


Berhasil scrape [18/200] - SPORT


Berhasil scrape [19/200] - SPORT


Berhasil scrape [20/200] - SPORT


KeyboardInterrupt: 

## **Langkah 3: Menampilkan Hasil**
Menampilkan beberapa sampel data teratas untuk memastikan hasil ekstraksi bersih.

In [10]:
# Tampilkan sampel data teratas (Sport) dan terbawah (Finance)
print("\n--- SAMPEL DATA SPORT ---")
display(df.head(3))

print("\n--- SAMPEL DATA FINANCE ---")
display(df.tail(3))



--- SAMPEL DATA SPORT ---


,url,category,title,date,text,title_length,text_length
0,https://sport.detik.com/sport-lain/d-8655067/k...,sport,Ketum KOI dan Menpora Mengukuhkan Tim Indonesi...,"Rabu, 09 Sep 2026 11:50 WIB",Ketua Umum Komite Olimpiade Indonesia (KOI) Ra...,70,2832
1,https://sport.detik.com/moto-gp/d-8654776/moto...,sport,MotoGP San Marino 2026: Jaga Puncak Klasemen B...,"Rabu, 09 Sep 2026 08:52 WIB",Persaingan titel juara dunia semakin ketat men...,73,1503
2,https://sport.detik.com/raket/d-8654666/ganda-...,sport,Ganda Campuran Indonesia Kembali Rombak Pemain,"Rabu, 09 Sep 2026 06:45 WIB",Sektor ganda campuran kembali mengalami peromb...,46,1442



--- SAMPEL DATA FINANCE ---


,url,category,title,date,text,title_length,text_length
197,https://finance.detik.com/berita-ekonomi-bisni...,finance,Kemenhub Ungkap Biang Kerok Kemacetan Panjang ...,"Selasa, 08 Sep 2026 08:10 WIB",Penanganan insentif terus dilakukan untuk meng...,64,3212
198,https://finance.detik.com/berita-ekonomi-bisni...,finance,Bandara Soetta-Husein Siap Beroperasi Lagi Usa...,"Selasa, 08 Sep 2026 07:38 WIB",InJourney Airports menjamin kesiapan operasion...,54,2702
199,https://finance.detik.com/moneter/d-8653044/pi...,finance,"Pinjaman Online Warga RI Makin Banyak, Tembus ...","Selasa, 08 Sep 2026 07:25 WIB",Otoritas Jasa Keuangan (OJK) mencatat total ut...,60,1530


## **Data Understanding**

In [11]:
import pandas as pd
import numpy as np

# 1. Menampilkan 5 baris pertama dan 5 baris terakhir data
print("--- 5 BARIS PERTAMA DATA ---")
display(df.head())
print("\n--- 5 BARIS TERAKHIR DATA ---")
display(df.tail())

# 2. Informasi Umum Dataset (Jumlah baris, kolom, tipe data, & memory usage)
print("\n--- INFORMASI UMUM DATASET ---")
df.info()

# 3. Pengecekan Missing Value (Nilai yang Kosong)
print("\n--- JUMLAH MISSING VALUE PER KOLOM ---")
missing_data = df.isnull().sum()
print(missing_data)

# 4. Distribusi Jumlah Data Berdasarkan Kategori
print("\n--- DISTRIBUSI KATEGORI (SPORT & FINANCE) ---")
category_counts = df['category'].value_counts()
print(category_counts)

# 5. Statistik Deskriptif untuk Panjang Karakter Teks & Judul
print("\n--- STATISTIK DESKRIPTIF PANJANG TEKS & JUDUL ---")
display(df[['title_length', 'text_length']].describe())

# 6. Pengecekan Duplikasi Data (Berdasarkan URL atau Judul)
print("\n--- JUMLAH DATA DUPLIKAT ---")
print(f"Duplikat berdasarkan URL: {df['url'].duplicated().sum()}")
print(f"Duplikat berdasarkan Judul: {df['title'].duplicated().sum()}")

--- 5 BARIS PERTAMA DATA ---


,url,category,title,date,text,title_length,text_length
0,https://sport.detik.com/sport-lain/d-8655067/k...,sport,Ketum KOI dan Menpora Mengukuhkan Tim Indonesi...,"Rabu, 09 Sep 2026 11:50 WIB",Ketua Umum Komite Olimpiade Indonesia (KOI) Ra...,70,2832
1,https://sport.detik.com/moto-gp/d-8654776/moto...,sport,MotoGP San Marino 2026: Jaga Puncak Klasemen B...,"Rabu, 09 Sep 2026 08:52 WIB",Persaingan titel juara dunia semakin ketat men...,73,1503
2,https://sport.detik.com/raket/d-8654666/ganda-...,sport,Ganda Campuran Indonesia Kembali Rombak Pemain,"Rabu, 09 Sep 2026 06:45 WIB",Sektor ganda campuran kembali mengalami peromb...,46,1442
3,https://sport.detik.com/sport-lain/d-8654641/a...,sport,Ambisi Morgan Holindo Jadi Juara Nasional Esha...,"Rabu, 09 Sep 2026 02:10 WIB","Pegokar muda Indonesia, Morgan Holindo, tak pu...",61,1629
4,https://sport.detik.com/raket/d-8654637/mens-w...,sport,Men's World Tennis Championship: Hari Baik unt...,"Rabu, 09 Sep 2026 01:10 WIB",Seri V Men's World Tennis Championship 2026 su...,57,1689



--- 5 BARIS TERAKHIR DATA ---


,url,category,title,date,text,title_length,text_length
195,https://finance.detik.com/berita-ekonomi-bisni...,finance,17 Perjalanan KRL Dibatalkan Imbas Perawatan A...,"Selasa, 08 Sep 2026 08:27 WIB",Penyesuaian jadwal Commuter Line Jabodetabek t...,63,2615
196,https://finance.detik.com/bursa-dan-valas/d-86...,finance,"Transisi Bisnis, TBS Energi (TOBA) Jadi Saham ...","Selasa, 08 Sep 2026 08:19 WIB",PT TBS Energi Utama Tbk (TOBA) menjadi perusah...,63,4728
197,https://finance.detik.com/berita-ekonomi-bisni...,finance,Kemenhub Ungkap Biang Kerok Kemacetan Panjang ...,"Selasa, 08 Sep 2026 08:10 WIB",Penanganan insentif terus dilakukan untuk meng...,64,3212
198,https://finance.detik.com/berita-ekonomi-bisni...,finance,Bandara Soetta-Husein Siap Beroperasi Lagi Usa...,"Selasa, 08 Sep 2026 07:38 WIB",InJourney Airports menjamin kesiapan operasion...,54,2702
199,https://finance.detik.com/moneter/d-8653044/pi...,finance,"Pinjaman Online Warga RI Makin Banyak, Tembus ...","Selasa, 08 Sep 2026 07:25 WIB",Otoritas Jasa Keuangan (OJK) mencatat total ut...,60,1530



--- INFORMASI UMUM DATASET ---
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   url           200 non-null    str  
 1   category      200 non-null    str  
 2   title         200 non-null    str  
 3   date          196 non-null    str  
 4   text          200 non-null    str  
 5   title_length  200 non-null    int64
 6   text_length   200 non-null    int64
dtypes: int64(2), str(5)
memory usage: 11.1 KB

--- JUMLAH MISSING VALUE PER KOLOM ---
url             0
category        0
title           0
date            4
text            0
title_length    0
text_length     0
dtype: int64

--- DISTRIBUSI KATEGORI (SPORT & FINANCE) ---
category
sport      100
finance    100
Name: count, dtype: int64

--- STATISTIK DESKRIPTIF PANJANG TEKS & JUDUL ---


,title_length,text_length
count,200.000000,200.000000
mean,59.515000,2371.555000
std,11.710914,1137.449371
min,20.000000,334.000000
25%,52.750000,1699.500000
50%,61.000000,2183.500000
75%,68.250000,2681.000000
max,80.000000,8169.000000



--- JUMLAH DATA DUPLIKAT ---
Duplikat berdasarkan URL: 0
Duplikat berdasarkan Judul: 0
